# Lab 20 · Prawdopodobieństwo i alarm

Przed wykonaniem tego laboratorium przeczytaj wykłady [D16.1 „Od częstości do prawdopodobieństwa”](../lectures/d16-frequency.html), [D16.2 „Zamień prawdopodobieństwo na alarm”](../lectures/d16-alarms.html) i [D16.3 „Zbuduj etykietę jutra i model logistyczny”](../lectures/d16-logistic.html).

Pamiętaj, że do tego labu potrzebne jest środowisko z labu 9: projekt practice z bibliotekami sekcji C. Uruchom Jupyter tak samo jak tam.

Pod opisem każdego zadania znajduje się komórka kodu do uzupełnienia, a pod nią komórka sprawdzenia. Jeżeli zadanie nie jest jeszcze rozwiązane, sprawdzenie zatrzyma się i wyświetli czerwony komunikat. To normalne: komunikat mówi, co trzeba poprawić.

Najpierw przewidź wynik, potem uruchom kod. Podpowiedzi otwieraj pojedynczo.

In [ ]:
from pathlib import Path
import sys
assert sys.version_info[:2] == (3, 12), "Wybierz kernel Pythona 3.12 z głównego README."
lab_dir = Path.cwd()
assert (lab_dir / "lab-20-prawdopodobienstwo-i-alarm.ipynb").is_file(), "Uruchom Jupyter z folderu labs."
print("Folder:", lab_dir)

In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError as error:
    raise AssertionError("Brakuje bibliotek sekcji C. Dodaj je w practice i uruchom Jupyter przez --project ../practice, zgodnie z głównym README.") from error
print("pandas", pd.__version__, "| numpy", np.__version__)

## Trzydzieści dni zdarzeń

Gotowa komórka wczytuje `data/d16-stock-events.csv`: jawnie syntetyczny szereg 30 dni jednego produktu. Kolumna `low_stock_evening` mówi, czy wieczorem zapas był niski, `promo` czy w danym dniu była planowana promocja, a `stockout` czy w ciągu dnia zabrakło towaru. Zera i jedynki są zdarzeniami, nie liczbą sztuk.

In [ ]:
events = pd.read_csv(lab_dir / "data" / "d16-stock-events.csv")
assert events.shape == (30, 4), "Oczekiwano 30 dni i czterech kolumn."
print(events.head())
print("Dni:", len(events), "| braki towaru:", int(events["stockout"].sum()))

## D16.1 · Częstość bazowa i warunkowa

Po wykładzie D16.1 w `base` zapisz udział dni z brakiem towaru wśród wszystkich dni. W `promo_days` zapisz liczbę dni promocji, w `conditional_promo` udział braków wśród dni promocji, a w `conditional_no_promo` udział braków wśród pozostałych dni. Przewidź, która częstość warunkowa będzie wyższa i zapisz liczebność grupy obok wyniku.

In [ ]:
base = None
promo_days = None
conditional_promo = None
conditional_no_promo = None

In [ ]:
import math
assert base is not None and math.isclose(float(base), 0.43333333333333335), "D16.1: częstość bazowa to średnia kolumny stockout."
assert promo_days == 10, "D16.1: policz dni z promo równym 1."
assert conditional_promo is not None and math.isclose(float(conditional_promo), 0.7), "D16.1: w częstości warunkowej mianownikiem jest liczba dni promocji, nie wszystkich dni."
assert conditional_no_promo is not None and math.isclose(float(conditional_no_promo), 0.3), "D16.1: druga grupa to dni bez promocji."
print("D16.1: baza", round(float(base), 3), "| przy promocji", round(float(conditional_promo), 3), "na", promo_days, "dniach | bez promocji", round(float(conditional_no_promo), 3))

<details><summary>Podpowiedź 1</summary>

Maska events["promo"] == 1 wybiera grupę; średnia stockout w tej grupie to częstość warunkowa.

</details>

<details><summary>Podpowiedź 2</summary>

Udział jednoczesnych promocji i braków wśród wszystkich dni to inna liczba niż częstość w grupie.

</details>

## D16.2 · Macierz pomyłek, dwie miary i koszt

Po wykładzie D16.2 dziesięć dni ma rzeczywiste zdarzenia `y_true` i wyniki modelu `scores`. Alarm zachodzi, gdy wynik jest nie mniejszy od progu, czyli `score >= threshold`, jak na wykładzie. Dla progu `threshold` zbuduj `confusion` jako słownik z polami `tp`, `fp`, `fn` i `tn`, a potem policz `precision` i `recall`. Następnie dla progów `candidates` policz `costs`: słownik próg → koszt, gdzie false positive kosztuje 1, a false negative 5. W `best_threshold` zapisz próg o najniższym koszcie. Najpierw wypełnij macierz na kartce.

In [ ]:
y_true = [1, 0, 1, 1, 0, 0, 1, 0, 0, 1]
scores = [0.9, 0.7, 0.6, 0.5, 0.45, 0.4, 0.35, 0.3, 0.2, 0.1]
threshold = 0.5
candidates = [0.3, 0.5, 0.7]
confusion = None
precision = None
recall = None
costs = None
best_threshold = None

In [ ]:
import math
assert isinstance(confusion, dict) and confusion == {'tp': 3, 'fp': 1, 'fn': 2, 'tn': 4}, "D16.2: alarm to score >= 0.5; policz tp, fp, fn i tn z par (zdarzenie, alarm)."
assert precision is not None and math.isclose(float(precision), 0.75), "D16.2: precision dzieli trafne alarmy przez wszystkie alarmy."
assert recall is not None and math.isclose(float(recall), 0.6), "D16.2: recall dzieli trafne alarmy przez wszystkie zdarzenia."
assert isinstance(costs, dict) and set(costs) == set(candidates) and all(math.isclose(float(costs[th]), wanted) for th, wanted in {0.3: 9, 0.5: 11, 0.7: 21}.items()), "D16.2: koszt progu to fp * 1 + fn * 5 dla alarmów przy tym progu."
assert best_threshold == 0.3, "D16.2: wybierz próg o najniższym koszcie wśród candidates."
print("D16.2: przy progu 0.5 precision", round(float(precision), 2), "recall", round(float(recall), 2), "| koszty", costs)

<details><summary>Podpowiedź 1</summary>

Lista alarmów to [s >= threshold for s in scores]; potem zlicz cztery kombinacje z y_true przez zip.

</details>

<details><summary>Podpowiedź 2</summary>

Napisz małą funkcję liczącą macierz dla dowolnego progu i użyj jej w pętli po candidates.

</details>

## D16.3 · Etykieta jutra i granica treningu

Po wykładzie D16.3 w `labelled` dodaj do `events` dwie kolumny: `tomorrow` jako `stockout` przesunięte o jeden dzień wstecz (`shift(-1)`), czyli zdarzenie następnego dnia, oraz `promo_tomorrow` jako `promo` przesunięte tak samo; plan promocji na jutro jest znany wieczorem. Ostatni dzień nie ma etykiety i pozostaje brakiem; nie wpisuj zera.

Model przygotowujesz wieczorem dnia 20. W `train_rows` zostaw wiersze o `day` najwyżej 19: ich etykiety opisują zdarzenia najpóźniej dnia 20. W `valid_rows` zostaw wiersze o `day` od 20 do 29; to dziesięć codziennie aktualizowanych prognoz na dni 21–30.

In [ ]:
labelled = None
train_rows = None
valid_rows = None

In [ ]:
assert isinstance(labelled, pd.DataFrame) and {"tomorrow", "promo_tomorrow"} <= set(labelled.columns), "D16.3: dodaj kolumny tomorrow i promo_tomorrow przez assign."
assert labelled["tomorrow"].iloc[-1] != labelled["tomorrow"].iloc[-1], "D16.3: ostatni dzień nie ma znanej etykiety; shift(-1) zostawia NaN."
assert labelled["tomorrow"].iloc[:-1].tolist() == events["stockout"].iloc[1:].tolist(), "D16.3: tomorrow w wierszu t to stockout z wiersza t + 1."
assert labelled["promo_tomorrow"].iloc[:-1].tolist() == events["promo"].iloc[1:].tolist(), "D16.3: promo_tomorrow to plan promocji na następny dzień."
assert isinstance(train_rows, pd.DataFrame) and len(train_rows) == 19 and train_rows["day"].max() == 19, "D16.3: trening to dni 1–19; wiersz 20 opisuje już dzień 21."
assert isinstance(valid_rows, pd.DataFrame) and len(valid_rows) == 10 and valid_rows["day"].min() == 20 and valid_rows["day"].max() == 29, "D16.3: walidacja to wiersze 20–29; wiersz 30 nie ma etykiety."
assert set(train_rows["tomorrow"].astype(int)) == {0, 1}, "D16.3: trening musi zawierać obie klasy."
print("D16.3: etykieta jutra zbudowana; trening kończy się przed chwilą prognozy.")

<details><summary>Podpowiedź 1</summary>

shift(-1) przesuwa wartości o jeden wiersz w górę; ostatni wiersz dostaje NaN.

</details>

<details><summary>Podpowiedź 2</summary>

Maski po kolumnie day: <= 19 dla treningu, między 20 i 29 dla walidacji.

</details>

## D16.3 · Model logistyczny i alarmy na walidacji

W `classifier` dopasuj `LogisticRegression()` na cechach `["low_stock_evening", "promo_tomorrow"]` z `train_rows` i etykiecie `train_rows["tomorrow"]` zamienionej na int. W `probabilities` zapisz listę prawdopodobieństw klasy 1 z `predict_proba` dla `valid_rows`. W `alarms` zapisz listę True/False: True, gdy prawdopodobieństwo jest nie mniejsze niż 0.5, a w `recall_valid` recall tych alarmów względem rzeczywistych etykiet walidacji. Dziesięć dni sprawdza przepływ, nie jakość metody.

In [ ]:
classifier = None
probabilities = None
alarms = None
recall_valid = None

In [ ]:
import math
from sklearn.linear_model import LogisticRegression
assert isinstance(classifier, LogisticRegression) and list(classifier.classes_) == [0, 1], "D16.3: dopasuj LogisticRegression na treningu z obiema klasami."
assert isinstance(probabilities, list) and len(probabilities) == 10 and np.allclose(probabilities, [0.5558190830741595, 0.31882518372633223, 0.31882518372633223, 0.7290197053933912, 0.17878223808692917, 0.31882518372633223, 0.5558190830741595, 0.7290197053933912, 0.5558190830741595, 0.17878223808692917], atol=1e-3), "D16.3: predict_proba(valid)[:, 1] daje prawdopodobieństwo zdarzenia; fit tylko na train_rows."
assert alarms == [p >= 0.5 for p in probabilities], "D16.3: alarm to prawdopodobieństwo co najmniej 0.5."
assert recall_valid is not None and math.isclose(float(recall_valid), 0.8), "D16.3: recall to trafne alarmy podzielone przez zdarzenia w walidacji."
print("D16.3: prawdopodobieństwa", [round(p, 2) for p in probabilities], "| recall przy 0.5:", round(float(recall_valid), 2))

<details><summary>Podpowiedź 1</summary>

Cechy to tabela z dwiema kolumnami; etykieta to kolumna tomorrow jako int.

</details>

<details><summary>Podpowiedź 2</summary>

predict_proba zwraca dwie kolumny; druga to klasa 1. Recall policz z par (etykieta, alarm) jak w D16.2.

</details>

## Sprawdź rezultat

Sprawdzenia przechodzą po restarcie kernela. Do przemyślenia: dlaczego częstość przy promocji liczona na dziesięciu dniach jest mniej pewna niż bazowa na trzydziestu? Czy próg 0.5 jest dobry przy koszcie 5 za pominięte zdarzenie? Dlaczego wiersz 20 nie może należeć do treningu modelu przygotowanego wieczorem dnia 20? Nie musisz spisywać odpowiedzi.